In [27]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


## Middleware

Middleware sits between the agent's core loop and its inputs/outputs, letting you intercept and modify behavior without changing the agent's underlying logic. Middleware is useful for the following:

- **Observability** — track agent behavior with logging, analytics, and debugging so you can see what the agent is doing and why.
- **Transformation** — rewrite prompts, filter or select which tools are available, and reformat outputs before they're returned.
- **Resilience** — add retries on failure, fallback models/tools, and early termination logic to stop runaway loops.
- **Safety** — enforce rate limits, guardrails, and PII detection/redaction to keep the agent's behavior within bounds.

### Summarization Middleware

As a conversation grows, the message history can exceed the model's context window or become expensive to send on every call. Summarization middleware solves this by automatically condensing older messages into a compact summary once a size threshold is hit, while keeping recent messages intact.

- **Trigger** — kicks in when the conversation history crosses a configured token/message limit.
- **Condense** — sends the older messages to an LLM to produce a short summary, replacing the originals.
- **Preserve recency** — keeps the most recent N messages verbatim so immediate context isn't lost.
- **Transparent to the agent** — the agent still sees a coherent history; it doesn't need to manage truncation itself.

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="gpt-5-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-mini",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [29]:
questions = [
    "What is 41 + 4?",
    "What is 2 - 9?",
    "What is 16 * 8?",
    "What is 9 + 4?",
    "What is 44 - 18?",
    "What is 6 * 19?",
    "What is 28 + 2?",
    "What is 2 - 3?",
]
len(questions)

8

In [30]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [31]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 41 + 4?', additional_kwargs={}, response_metadata={}, id='8ea7e530-677f-403b-b8be-3b4c9da26180'), AIMessage(content='45', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 14, 'total_tokens': 24, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQrysbvEKWqBywFaQl4EqnWWKOWBn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c8a6-4a91-7002-94b9-f8fbda8f8fce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24, 'input

## Token size

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="gpt-5-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-mini",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [33]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~209 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='78645a0b-8c05-486f-8627-120009c53e76'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 134, 'total_tokens': 286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQrzWeOH6OVlPKBtWaPIKrci162pi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8a6-ed8f-7d70-8bd2-37bf399e75ef-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_sJBlCVhsmFTHpnbYmGxaNw

## Fraction

In [34]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-5-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~135 tokens (0.1055%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='07ae4dd4-55c3-4b39-ad84-18eb38a8bad1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 125, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQs2JyOdNSA7DOSaz6r4RpSXa3mpF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c8a9-9064-7002-833e-7d9c5449aa1a-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_WKEZMMwH6lYWMe2RKOGFKXT

### Model Fallback Middleware

If the primary model call fails (e.g. rate limit, timeout, outage), this middleware automatically retries the request with the next model in the fallback list, in order, until one succeeds.

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware

agent = create_agent(
    model="openai:gpt-5.0-with-typo",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            "openai:gpt-4.0-mini-with-typo",
            "gpt-5-mini",
        ),
    ],
)

In [26]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is 2+2?"}]})
last_message = response["messages"][-1]
print(last_message.content)
print(last_message.response_metadata.get("model_name"))


4
gpt-5-mini-2025-08-07
